# Domain Matching

This notebook is a demo using the python package bacpipe to perform calculation in the latent space (i.e. embeddings space).

**contents**
1. Install bacpipe
2. Setup & Configuration
3. Compute the embeddings of the audio
4. Train and test a classifier using embeddings


---
## 1. Imports & Working Directory

Import the packages needed for the analysis and set the working directory to the repo root.


In [1]:
# to run successfully the packages for jupyter notebook need to be installed:
# pip install ipykernel ipython jupyter_bokeh

# if troubles with numpy 2 => uninstall numpy and install numpy 1.24.6, 
# then install pillow, pyaml, bokeh

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path

# interactive display of audio in Jupyter notebooks
from IPython.display import display
from IPython.display import Audio

# library for audio and music analysis
import librosa as lb

# load the specific package
import bacpipe


In [2]:
# get the path to the folder where the notebook is located
notebook_path = Path().resolve()

# get the name of the current notebook file
notebook_file = 'domain_matching'

# get the directoy where the notebook is located
notebook_dir = notebook_path.name

# working directory 
wd_path = notebook_path.parent

# set the audio directory to the folder 
# audio files directory
bacpipe.config.audio_dir = str( wd_path / 'data' / notebook_file) 

# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(wd_path / 'results')

## 2. Download and unzip the data

In [ ]:
# download the audio files, unzip the folders
os.chdir(wd_path)
os.chdir("data")
if not os.path.exists("domain_matching"):
    !unzip domain_matching.zip
    # # remove *.zip files
    # try :
    #     os.remove("domain_matching.zip")
    # except FileNotFoundError:
    #     print("File not found. Skipping deletion.")
os.chdir("../")

## 3. Compute the embeddings of the audio

* Extract the list of audio
* Generate an long raw audio, a long audio mixed with a background
* Compute the embeddings (3s chunks) for each long audio


You can load and preprocess audio and pass it directly to an
`Embedder` — useful when you need custom loading logic or want to process audio
that isn't stored on disk.

Here we load all test audio files with `librosa`, concatenate the samples into a
single long 1D numpy array, and pass it to `Embedder.generate_embeddings_from_audio_array`.
The method windows the audio into segments of the model's input length and returns
the embedding of each window. `bacpipe.ensure_models_exist` guarantees that the
model checkpoint is present locally, downloading it from the Hugging Face Hub on
first use.



### 3.1 Extract the list of audio: case of bird species

In [4]:
# load and concatenate audio, then pass it directly to an Embedder
list_audio_files = bacpipe.get_audio_files(
    str(Path(bacpipe.config.audio_dir) / "species"), return_type="str"
)

# convert loader_obj.files into a dataframe with columns 'file_path' and 'file_name' and 'label'
# which is the parent folder name of the audio file)
df_species = pd.DataFrame(
    {
        "file_path": [f for f in list_audio_files],
        "file_name": [(Path(f).name) for f in list_audio_files],
        "label": [Path(f).parent.name for f in list_audio_files],
    }
)
# print the unique values of the label column with a list corresponding to the range of rows where the label is equal to the unique value
print("Unique values of the label column:")
for label in df_species["label"].unique():
    print(f"{label}: {list(df_species[df_species['label'] == label].index)}")



finding audio files: 0it [00:00, ?it/s]
Found 0 number of audio files.


AssertionError: No audio files found in audio_dir.

### 3.2 Extract the list of audio: case of background sound

In [ ]:
# load the background sound
list_background_files = bacpipe.get_audio_files(
    str(Path(bacpipe.config.audio_dir) / "background"), return_type="str"
)

# convert loader_obj.files into a dataframe with columns 'file_path' and 'file_name' and 'label'
# which is the parent folder name of the audio file)
df_background = pd.DataFrame(
    {
        "file_path": [f for f in list_background_files],
        "file_name": [(Path(f).name) for f in list_background_files],
        "label": [Path(f).parent.name for f in list_background_files],
    }
)

# print the unique values of the label column with a list corresponding to the range of rows where the label is equal to the unique value
print("Unique values of the label column:")
for label in df_background["label"].unique():
    print(f"{label}: {list(df_background[df_background['label'] == label].index)}")


finding audio files: 14it [00:00, 37282.70it/s]
Found 10 number of audio files.


Unique values of the label column:
Ambient Sound: [0, 1, 2]
Great Green Bush-Cricket: [3]
Light Rain: [4, 5]
Tropical Rainforest: [6, 7, 8, 9]


In [ ]:
# load the background sound
# Select the index of the background sound you want to use 
# (see the printed list of unique values above)
BCK_ID = 3 # choose 8
# Set the SAMPLE_RATE to 48000 Hz and the DURATION to 3 seconds
SAMPLE_RATE = 48000 # birdnet:48000 birdnet_v3:32000
DURATION = 3
background_sound, _ = lb.load(
    df_background.iloc[BCK_ID]["file_path"],
    sr=SAMPLE_RATE,
    duration=DURATION,
)

# play the background sound
Audio(background_sound, rate=SAMPLE_RATE)

In [ ]:
# concatenate the audio files and add background sound
audio_concatenate = []
audio_chunks = []
audio_concatenate_mixed = []
audio_chunks_mixed = []
for file in list_audio_files:
    aud, sr = lb.load(file, sr=SAMPLE_RATE, duration=DURATION)
    # zero pad the audio to DURATION seconds
    if len(aud) < DURATION * sr:
        print(f"Zero padding {file} from {len(aud)} to {DURATION * sr} samples.")
        aud = np.pad(aud, DURATION * sr - len(aud), "constant")
    audio_chunks.append(aud)
    audio_concatenate.extend(aud)
    # add background
    audio_mixed = aud * 0.005 + background_sound * 0.995
    audio_chunks_mixed.append(audio_mixed)
    audio_concatenate_mixed.extend(audio_mixed)

df_species["audio"] = audio_chunks
df_species["audio_mixed"] = audio_chunks_mixed


In [ ]:
# Auto-install maad if missing
import importlib.util

if importlib.util.find_spec("maad") is None:
    print("maad not found. Installing...")
    os.system("pip install scikit-maad")

# display the spectrogram of the audio (before mixing with background sound)
from maad import sound, util

label = 'Eurasian Blackbird'
index = df_species[df_species['label'] == label].sample().index
s = df_species['audio'][index].iloc[0]

# display the spectrogram of the first 3 seconds of audio
Sxx, tn, fn, ext = sound.spectrogram(s, fs=SAMPLE_RATE) 
fig, ax = plt.subplots(2,1, figsize=(10,6))
util.plot_wave(s, SAMPLE_RATE, ax=ax[0])
util.plot_spectrogram(Sxx, ext, db_range=96, gain=30, colorbar=False, ax=ax[1])

# play the sound
Audio(s, rate=SAMPLE_RATE)

/home/haupert/miniconda3/envs/bacpipe/lib/python3.11/site-packages/maad/util/miscellaneous.py:413: RuntimeWarning: divide by zero encountered in log10
  y = 10*log10(x)   # take log


In [ ]:
# display the spectrogram of the mixed audio
s = df_species['audio_mixed'][index].iloc[0]  # get the first audio chunk

# display the spectrogram of the first 3 seconds of audio
Sxx, tn, fn, ext = sound.spectrogram(s, fs=SAMPLE_RATE) 
fig, ax = plt.subplots(2,1, figsize=(10,6))
util.plot_wave(s, SAMPLE_RATE, ax=ax[0])
util.plot_spectrogram(Sxx, ext, db_range=96, gain=30, colorbar=False, ax=ax[1])

# play the sound
Audio(s, rate=SAMPLE_RATE)

Using the `Embedder` class, we can compute the embeddings of the long audio files. The embeddings are computed in chunks of 3 seconds, which allows for efficient processing and analysis of the audio data.

In [ ]:
# display the list of available models
display(bacpipe.supported_models)

['audiomae',
 'audioprotopnet',
 'avesecho_passt',
 'aves_especies',
 'bat',
 'batdetect2_clip_avg',
 'batdetect2_dets_avg',
 'beats',
 'birdaves_especies',
 'biolingual',
 'birdnet_v3',
 'birdnet',
 'birdmae',
 'convnext_birdset',
 'hbdet',
 'insect66',
 'insect459',
 'mix2',
 'naturebeats',
 'perch_bird',
 'perch_v2',
 'protoclr',
 'rcl_fs_bsed',
 'surfperch',
 'google_whale',
 'vggish']

In [ ]:
# select the model
MODEL = 'birdnet' # birdnet birdnet_v3

# check if the model exists, if not, download it
bacpipe.ensure_models_exist(bacpipe.settings.model_base_path, [MODEL])

# load the model and create an embedder object
embed_obj = bacpipe.Embedder(MODEL)

# Compute the embeddings for the concatenated audio and the mixed audio
embeds = embed_obj.generate_embeddings_from_audio_array(np.array(audio_concatenate))
embeds_mixed = embed_obj.generate_embeddings_from_audio_array(np.array(audio_concatenate_mixed))
embed_background = embed_obj.generate_embeddings_from_audio_array(background_sound)

# subtract the background sound embeddings from the mixed audio embeddings
embeds_corrected = np.array(embeds_mixed) - np.array(embed_background)


Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

2026-09-15 12:59:16.290080: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-15 12:59:16.700133: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
birdnet checkpoint does not exists. Downloading the model from https://huggingface.co/datasets/vskode/bacpipe_models/blob/main/birdnet/birdnet.tar.xz



birdnet/birdnet.tar.xz:   0%|          | 0.00/52.9M [00:00<?, ?B/s]

Using device='cpu'
2026-09-15 12:59:20.520464: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-09-15 12:59:20.520486: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2026-09-15 12:59:20.520490: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2026-09-15 12:59:20.520492: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-09-15 12:59:20.520494: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: risoux
2026-09-15 12:59:20.520500: I external/local_xla/xla/stream_e

Using the embeddings, we can train a classifier to predict the label (species) of the audio files. The classifier can be trained on a labeled dataset of audio files with known domains (i.e. no background to little background noise), and then used to predict the labels of the new audio files with never seen domain (such as rain, wind, or even other habitat (such as rainforest).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Train a classifier using the raw embedings without the background sound
# X is the embeddings and y is the labels
X = embeds
y = df_species["label"].to_numpy()

custom_classifier = KNeighborsClassifier(
    n_neighbors=1, 
    metric="cosine")

custom_classifier.fit(X, y)


,n_neighbors,1
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'cosine'
,metric_params,None
,n_jobs,None


In [ ]:
# compute the accuracy of the custom classifier on the embeddings of the mixed audio
y_pred = custom_classifier.predict(embeds_mixed)
y_true = df_species["label"].to_numpy()

# compute the accuracy
accuracy = np.mean(y_pred == y_true)
print(f'Accuracy on mixed audio (never seen domains => domain mismatch): {accuracy:.2f}')

# compute the accuracy of the custom classifier on the embeddings of the mixed audio
y_pred = custom_classifier.predict(embeds_corrected)
y_true = df_species["label"].to_numpy()

# compute the accuracy
accuracy = np.mean(y_pred == y_true)
print(f'Accuracy on corrected audio (subtracting the background sound): {accuracy:.2f}')

Accuracy on mixed audio (never seen domains => domain mismatch): 0.78
Accuracy on corrected audio (subtracting the background sound): 0.92
